
<div  style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://raw.githubusercontent.com/derar-alhussein/Databricks-Certified-Data-Engineer-Associate/main/Includes/images/bookstore_schema.png" alt="Databricks Learning" style="width: 600">
</div>

In [0]:
%run ../Includes/Copy-Datasets

##Exploring The Source Directory

In [0]:
files = dbutils.fs.ls(f"{dataset_bookstore}/orders-raw")
display(files)

##Auto Loader

In [0]:
(
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format","parquet")
    .option("cloudFiles.schemaLocation",f"{checkpoints_bookstore}/order_raw")
    .load(f"{dataset_bookstore}/orders-raw")
    .createOrReplaceTempView("orders_raw_temp")
)

##Enriching Raw Data

In [0]:
%sql
create or replace temporary view order_tmp as (
    select *,current_timestamp() as arrival_time,input_file_name() as source_file 
    from orders_raw_temp
)

In [0]:
import time
order_tmp_df = spark.sql("select * from order_tmp")
display(order_tmp_df, checkpointLocation=f"{checkpoints_bookstore}/tmp/orders_{time.time()}")

##Creating Bronze Table

In [0]:
(
    spark.table("order_tmp")
        .writeStream
        .format("delta")
        .option("checkpointLocation", f"{checkpoints_bookstore}/orders_bronze")
        .outputMode("append")
        .toTable("orders_bronze")
)

In [0]:
%sql
select count(*) from orders_bronze

In [0]:
load_new_data()

##Creating Static lookup Table

In [0]:
(
    spark.read
    .format("json")
    .load(f"{dataset_bookstore}/customers-json")
    .createOrReplaceTempView("customers_lookup")
)

In [0]:
%sql
select * from customers_lookup

##Creating Silver layer

In [0]:
(
    spark.readStream
    .table("orders_bronze")
    .createOrReplaceTempView("orders_bronze_tmp")
)

In [0]:
%sql
create or replace temporary view orders_enriched_tmp as(
    select order_id,quantity,o.customer_id,c.profile:first_name as f_name,c.profile:last_name as l_name,
        cast(from_unixtime(order_timestamp,'yyyy-MM-dd HH:mm:ss') as timestamp)
        as order_timestamp,books
        from orders_bronze_tmp o
        inner join customers_lookup c
        on o.customer_id=c.customer_id
        where quantity>0
)

In [0]:
(
    spark.table("orders_enriched_tmp")
        .writeStream
        .format("delta")
        .option("checkpointLocation", f"{checkpoints_bookstore}/orders_silver")
        .outputMode("append")
        .toTable("orders_silver")
)

In [0]:
%sql
SELECT * FROM orders_silver

In [0]:
%sql
SELECT COUNT(*) FROM orders_silver

In [0]:
load_new_data()

##Creating Gold Table

In [0]:
(
    spark.readStream
    .table("orders_silver")
    .createOrReplaceTempView("orders_silver_tmp")
)

In [0]:
%sql
create or replace temp view daily_customer_books_tmp as(
    select customer_id,f_name,l_name,date_trunc("DD",order_timestamp) as order_date,sum(quantity) as books_counts
    from orders_silver_tmp
    group by customer_id,f_name,l_name,date_trunc("DD",order_timestamp)
)

In [0]:
(spark.table("daily_customer_books_tmp")
      .writeStream
      .format("delta")
      .outputMode("complete")
      .option("checkpointLocation", f"{checkpoints_bookstore}/daily_customer_books")
      .trigger(availableNow=True)
      .toTable("daily_customer_books"))

In [0]:
%sql
SELECT * FROM daily_customer_books

In [0]:
load_new_data(all=True)

##Stopping active streams

In [0]:
for s in spark.streams.active:
    print("Stopping stream: " + s.id)
    s.stop()
    s.awaitTermination()